# Actividad — Del dato crudo al dato de negocio

### Construir las capas Bronze, Silver y Gold en Unity Catalog, y gobernarlas

| | |
|---|---|
| Dónde corre | Databricks **Free Edition** (cómputo serverless) |
| Lenguajes | Python y SQL |
| Duración | Entre 45 y 60 minutos |
| Requisito | Haber visto la Sesión 1 (catálogo, esquema, tabla Delta, volúmenes) |

En la presentación vimos la arquitectura **Medallion** como un dibujo: tres cajas,
bronce, plata y oro. Aquí la construimos de verdad, con un dataset que vamos a
descargar de internet y que **viene sucio**, como vienen los datos reales.

**Lo que vas a construir:**

```
   internet                 workspace  (catálogo)
      │                           │
      ▼           ┌───────────────┼───────────────┐
  volumen  ──▶  cine_bronze   cine_silver     cine_gold
 (aterrizaje)   tal como        limpio y       agregado para
                 llegó        con contrato      el negocio
```

**La idea de fondo:** cada capa tiene un trabajo y solo uno. Bronze guarda lo que
llegó sin opinar. Silver decide qué es válido. Gold responde preguntas de negocio.
Y Unity Catalog gobierna las tres: quién entra, qué significa cada columna y qué
pasó con el dato.

---
## 1. Preparar el terreno

Tres capas, tres esquemas. Separarlas no es manía de orden: es lo que permite dar
permisos distintos a cada una. Un analista de negocio trabaja contra `cine_gold` y
no necesita —ni debe— ver el crudo.

En Free Edition solo existe el catálogo `workspace`, así que las tres capas viven
dentro de él. En una cuenta de empresa lo normal es un catálogo por ambiente
(`desarrollo`, `produccion`) con los tres esquemas dentro de cada uno.

*Los creamos desde Python, en un bucle, porque son tres variaciones de lo mismo:*

In [0]:
ESQUEMAS = {
    "cine_bronze": "Capa Bronze: datos tal como llegaron del origen, sin transformar",
    "cine_silver": "Capa Silver: datos limpios, tipados y con reglas de calidad",
    "cine_gold":   "Capa Gold: tablas agregadas listas para consumo de negocio",
}

for nombre, descripcion in ESQUEMAS.items():
    spark.sql(f"""
        CREATE SCHEMA IF NOT EXISTS workspace.{nombre}
        COMMENT '{descripcion}'
    """)
    print("Esquema listo: workspace." + nombre)

*Fíjate en que cada esquema lleva su `COMMENT`.* No es decoración: ese texto es
lo que aparece en el buscador del catálogo y lo que lee Genie para entender de qué
va cada cosa. **Documentar es la forma más barata de gobierno**, y es la que casi
nadie hace.

In [0]:
%sql
SELECT schema_name, comment
FROM workspace.information_schema.schemata
WHERE schema_name LIKE 'cine_%'
ORDER BY schema_name

Y el volumen donde va a aterrizar el archivo. Un volumen es el lugar gobernado
para archivos que no son tablas; recuerda que en Free Edition **DBFS no existe**.

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.cine_bronze.aterrizaje
COMMENT 'Zona de aterrizaje: archivos originales descargados del origen'

---
## 2. Traer el dato de internet

El dataset son **3 201 películas** con su recaudación, presupuesto, género,
estudio y calificaciones de IMDB y Rotten Tomatoes. Está publicado como JSON en el
repositorio público `vega-datasets`.

Lo descargamos con Python puro y lo dejamos **sin tocar** en el volumen. Ese
archivo original es el que nos permitiría reconstruirlo todo si algo sale mal más
adelante.

In [0]:
import urllib.request

URL = "https://raw.githubusercontent.com/vega/vega-datasets/main/data/movies.json"
DESTINO = "/Volumes/workspace/cine_bronze/aterrizaje/movies.json"

with urllib.request.urlopen(URL, timeout=60) as respuesta:
    contenido = respuesta.read()

with open(DESTINO, "wb") as f:
    f.write(contenido)

print(f"Descargados {len(contenido):,} bytes")
print("Guardado en:", DESTINO)

*Comprueba que el archivo está donde debe:*

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/cine_bronze/aterrizaje"))

---
## 3. Capa Bronze: guardar lo que llegó

La regla de Bronze es una sola: **no opines sobre el dato**. No corriges, no
filtras, no descartas columnas. Si el origen manda basura, Bronze guarda la basura.

Lo único que sí se añade son **metadatos de ingesta**: de qué archivo salió cada
fila y cuándo entró. Es lo que te permite, seis meses después, responder "¿de dónde
salió este número?".

*Primero lo leemos. `multiLine` hace falta porque el JSON es un array formateado en
varias líneas, no un objeto por línea.*

In [0]:
df_crudo = (spark.read
    .option("multiLine", True)
    .json(DESTINO))
df_crudo.printSchema()
print('Filas:',df_crudo.count())
df_crudo.show(10)

**Mira los nombres de las columnas:** `US Gross`, `Major Genre`, `Running Time
min`. Vienen con espacios, porque así los mandó el origen.

Eso es un problema concreto: **Delta no admite espacios en los nombres de columna**.
Así que saneamos los nombres —y solo los nombres—. Los valores entran intactos.

In [0]:
import re
from pyspark.sql import functions as F

def sanear(nombre):
    """US Gross -> us_gross   |   Running Time min -> running_time_min"""
    return re.sub(r"[^0-9a-zA-Z]+", "_", nombre).strip("_").lower()

nombres = [sanear(c) for c in df_crudo.columns] + ["_archivo_origen", "_ingestado_en"]

df_bronze = (df_crudo
    .withColumn("_archivo_origen", F.col("_metadata.file_path"))
    .withColumn("_ingestado_en",   F.current_timestamp())
    .toDF(*nombres))

print(df_bronze.columns)

> La alternativa a renombrar es activar *column mapping* en la tabla
> (`delta.columnMapping.mode = 'name'`), que sí permite espacios y acentos. Sirve
> cuando estás obligado a conservar el nombre original exacto.

*Y lo escribimos como tabla Delta:*

In [0]:
(df_bronze.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable("workspace.cine_bronze.peliculas_crudo"))

print("Tabla creada: workspace.cine_bronze.peliculas_crudo")

In [0]:
%sql
SELECT * FROM workspace.cine_bronze.peliculas_crudo LIMIT 10

*Observa las dos últimas columnas:* `_archivo_origen` y `_ingestado_en`. Son
nuestras, no del dataset. Esa es la **trazabilidad** que distingue una tabla Bronze
de un archivo tirado en un bucket.

---
## 4. ¿Qué tan sucio está? El perfilado

Antes de limpiar hay que mirar. Este paso casi siempre se salta, y es el que decide
qué reglas tiene sentido escribir.

*Primero, qué porcentaje de nulos tiene cada columna:*

In [0]:
tabla = spark.table("workspace.cine_bronze.peliculas_crudo")
total = tabla.count()

nulos = tabla.select([
    F.round(100 * F.sum(F.col(c).isNull().cast("int")) / total, 1).alias(c)
    for c in tabla.columns
])

display(nulos)

*Lee esa fila como porcentajes.* Hay dos columnas que están más vacías que
llenas: `us_dvd_sales` (82 %) y `running_time_min` (62 %). Una columna así no es un
dato con huecos: es una columna que no existe. En Silver descartaremos la primera y
conservaremos la segunda como opcional.

**Ahora la parte divertida.** Las fechas vienen como texto con formato
`"Jun 12 1998"`. Veamos en qué años se estrenaron estas películas:

In [0]:
%sql
SELECT right(release_date, 4) AS anio, count(*) AS peliculas
FROM workspace.cine_bronze.peliculas_crudo
GROUP BY ALL
ORDER BY anio DESC
LIMIT 15

*Y ahí está el primer regalo del mundo real:* hay películas estrenadas en **2046,
2044, 2043**… Son películas de los años cuarenta cuyo año de dos dígitos se
interpretó mal en algún punto de la cadena. `Gone with the Wind` figura estrenada
en 2039.

Nadie las va a arreglar por nosotros. Nuestro trabajo es **detectarlas y no dejar
que contaminen el negocio**.

*Segundo regalo: títulos que se repiten.*

In [0]:
%sql
SELECT title, count(*) AS veces
FROM workspace.cine_bronze.peliculas_crudo
GROUP BY title
HAVING count(*) > 1
ORDER BY veces DESC, title
LIMIT 10

Hay 24 títulos repetidos. **Y casi ninguno es un duplicado de verdad:**

In [0]:
%sql
SELECT title, release_date, director, production_budget, worldwide_gross
FROM workspace.cine_bronze.peliculas_crudo
WHERE title = 'Casino Royale'
ORDER BY release_date

Son dos películas distintas: la de 1967 con John Huston y la de 2006 con Daniel
Craig. Remakes, no duplicados.

> **La lección más cara de la limpieza de datos:** si hubiéramos deduplicado por
> título, habríamos borrado dos docenas de películas reales. La clave de negocio
> aquí no es el título: es **título + fecha de estreno**.

---
## 5. Capa Silver: tipar, nombrar y decidir

Silver hace tres cosas:

1. **Tipos de verdad** — las fechas dejan de ser texto, los importes dejan de ser
   genéricos.
2. **Nombres de negocio** — quien consulte esta tabla habla español y habla de
   negocio, no de columnas de un JSON ajeno.
3. **Reglas de calidad** — se decide qué fila es válida y qué fila no.

*Empezamos por las dos primeras:*

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW peliculas_tipadas AS
SELECT
  title                                       AS titulo,
  try_to_date(release_date, 'MMM dd yyyy')    AS fecha_estreno,
  major_genre                                 AS genero,
  mpaa_rating                                 AS clasificacion,
  distributor                                 AS estudio,
  director                                    AS director,
  creative_type                               AS tipo_creativo,
  source                                      AS origen_historia,
  CAST(running_time_min AS INT)               AS duracion_min,
  CAST(production_budget AS BIGINT)           AS presupuesto,
  nullif(CAST(us_gross AS BIGINT), 0)         AS recaudacion_eeuu,
  nullif(CAST(worldwide_gross AS BIGINT), 0)  AS recaudacion_mundial,
  CAST(imdb_rating AS DOUBLE)                 AS calificacion_imdb,
  CAST(imdb_votes AS BIGINT)                  AS votos_imdb,
  CAST(rotten_tomatoes_rating AS INT)         AS calificacion_rt,
  _ingestado_en
FROM workspace.cine_bronze.peliculas_crudo

Tres decisiones que conviene explicar en voz alta:

- **`try_to_date` en vez de `to_date`.** Las funciones `try_*` devuelven `NULL`
  cuando no pueden convertir, en lugar de reventar el pipeline. En una carga
  nocturna de verdad, esa diferencia es dormir o no dormir.
- **`nullif(..., 0)` en las recaudaciones.** Aquí un cero no significa "no recaudó
  nada": significa "no lo sabemos". Dejarlo como cero hundiría cualquier promedio
  que calculemos después.
- **`us_dvd_sales` no aparece.** Estaba vacía en el 82 % de las filas; arrastrarla
  solo invita a que alguien la use sin mirar.

*Comprobemos que las fechas ya son fechas:*

In [0]:
%sql
SELECT titulo, fecha_estreno, year(fecha_estreno) AS anio, presupuesto
FROM peliculas_tipadas
ORDER BY fecha_estreno DESC
LIMIT 5

### 5.1 · Las reglas de calidad, escritas una sola vez

En lugar de filtrar a ciegas con un `WHERE`, marcamos **por qué** se rechaza cada
fila. Así el rechazo se puede explicar, y alguien puede corregirlo.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW peliculas_evaluadas AS
SELECT *,
  array_compact(array(
    CASE WHEN titulo IS NULL                  THEN 'titulo nulo' END,
    CASE WHEN fecha_estreno IS NULL           THEN 'fecha ilegible' END,
    CASE WHEN fecha_estreno > current_date()  THEN 'fecha futura' END,
    CASE WHEN presupuesto IS NULL
           OR presupuesto <= 0                THEN 'presupuesto no positivo' END
  )) AS motivos_rechazo
FROM peliculas_tipadas

In [0]:
%sql
select * from peliculas_evaluadas

*`array_compact` quita los `NULL` del array, así que una fila sana termina con una
lista vacía.* Veamos el recuento por motivo:

In [0]:
%sql
SELECT motivo, count(*) AS filas
FROM (
  SELECT explode(motivos_rechazo) AS motivo
  FROM peliculas_evaluadas
  WHERE size(motivos_rechazo) > 0
)
GROUP BY motivo
ORDER BY filas DESC

In [0]:
%sql
  SELECT explode(motivos_rechazo) AS motivo,
  FROM peliculas_evaluadas
  WHERE size(motivos_rechazo) > 0


Deberías ver **18 filas rechazadas**: 16 con fecha futura, una sin título y una
sin presupuesto.

> El número de "fecha futura" depende del día en que ejecutes esto, porque la regla
> compara contra `current_date()`. Las películas de los años cuarenta mal fechadas
> seguirán cayendo aquí durante muchos años.

### 5.2 · Cuarentena: los rechazos no se borran

Esta es la diferencia entre un pipeline serio y un `WHERE`. Las filas que no pasan
van a una tabla aparte, con su motivo y su hora. Mañana alguien las revisa.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.cine_silver.peliculas_cuarentena
COMMENT 'Filas que no superaron las reglas de calidad de la capa Silver'
AS
SELECT *, current_timestamp() AS _cuarentena_en
FROM peliculas_evaluadas
WHERE size(motivos_rechazo) > 0

In [0]:
%sql
SELECT titulo, fecha_estreno, presupuesto, motivos_rechazo
FROM workspace.cine_silver.peliculas_cuarentena
ORDER BY fecha_estreno DESC

*Ahí están, con nombre y apellido:* `Gone with the Wind`, `Fantasia`,
`The Best Years of Our Lives`. Clásicos de los cuarenta con la fecha rota.

### 5.3 · La tabla Silver

Ahora sí, las filas buenas. Y deduplicamos por la clave de negocio correcta
—**título + fecha de estreno**—, quedándonos con la de mayor recaudación si hubiera
empate.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.cine_silver.peliculas
COMMENT 'Películas limpias y tipadas. Una fila por título y fecha de estreno'
AS
SELECT * EXCEPT (motivos_rechazo, _fila)
FROM (
  SELECT *,
    row_number() OVER (
      PARTITION BY titulo, fecha_estreno
      ORDER BY recaudacion_mundial DESC NULLS LAST) AS _fila
  FROM peliculas_evaluadas
  WHERE size(motivos_rechazo) = 0
)
WHERE _fila = 1

In [0]:
%sql
SELECT * FROM workspace.cine_silver.peliculas
ORDER BY recaudacion_mundial DESC
LIMIT 10

### 5.4 · Cuadrar las cuentas

Todo lo que entró tiene que estar en algún sitio: o pasó a Silver, o está en
cuarentena. Si la suma no cuadra, perdimos datos por el camino y hay que averiguar
dónde.

In [0]:
%sql
SELECT
  (SELECT count(*) FROM workspace.cine_bronze.peliculas_crudo)      AS bronze,
  (SELECT count(*) FROM workspace.cine_silver.peliculas)            AS silver,
  (SELECT count(*) FROM workspace.cine_silver.peliculas_cuarentena) AS cuarentena,
  (SELECT count(*) FROM workspace.cine_silver.peliculas)
    + (SELECT count(*) FROM workspace.cine_silver.peliculas_cuarentena) AS suma_comprobacion

*Debe dar 3 201 = 3 183 + 18.* Si cuadra, el `row_number` no eliminó ningún
duplicado —y eso **confirma** lo que sospechábamos: los 24 títulos repetidos eran
remakes legítimos.

Esta celda de reconciliación es la que conviene dejar corriendo en producción. Es
barata y detecta el 80 % de los desastres silenciosos.

---
## 6. El contrato de la tabla Silver

Las reglas del paso anterior las aplicó *el pipeline*. Nada impide que mañana
alguien inserte una fila mala por otro camino.

Un **contrato de datos** se escribe en la tabla, no en el código: a partir de aquí
la tabla misma rechaza lo que no cumple, venga de donde venga.

In [0]:
CONTRATO = [
    "ALTER COLUMN titulo SET NOT NULL",
    "ALTER COLUMN fecha_estreno SET NOT NULL",
    "ADD CONSTRAINT presupuesto_positivo CHECK (presupuesto > 0)",
    "ADD CONSTRAINT duracion_razonable "
    "  CHECK (duracion_min IS NULL OR duracion_min BETWEEN 1 AND 500)",
    "ADD CONSTRAINT imdb_en_rango "
    "  CHECK (calificacion_imdb IS NULL OR calificacion_imdb BETWEEN 0 AND 10)",
]

for clausula in CONTRATO:
    spark.sql(f"ALTER TABLE workspace.cine_silver.peliculas {clausula}")
    print("Aplicado:", clausula.split("CHECK")[0].strip())

Y la clave primaria. En Unity Catalog es **informativa**: documenta cuál es la
clave de negocio y la aprovechan las herramientas de BI, pero Databricks no la
verifica fila a fila.

In [0]:
%sql
ALTER TABLE workspace.cine_silver.peliculas
ADD CONSTRAINT pk_peliculas PRIMARY KEY (titulo, fecha_estreno)

*Las restricciones `CHECK` viven en las propiedades de la tabla Delta:*

In [0]:
%sql
SHOW TBLPROPERTIES workspace.cine_silver.peliculas

*Y la clave primaria queda registrada en el catálogo, donde cualquier herramienta
puede leerla:*

In [0]:
%sql
SELECT constraint_name, constraint_type
FROM workspace.information_schema.table_constraints
WHERE table_schema = 'cine_silver' AND table_name = 'peliculas'

### 6.1 · Comprobemos que el contrato se cumple

Intentamos meter una película con presupuesto negativo. **Debe fallar.**

In [0]:
mala = """
INSERT INTO workspace.cine_silver.peliculas
SELECT titulo, fecha_estreno, genero, clasificacion, estudio, director,
       tipo_creativo, origen_historia, duracion_min,
       -1 AS presupuesto,
       recaudacion_eeuu, recaudacion_mundial, calificacion_imdb,
       votos_imdb, calificacion_rt, _ingestado_en
FROM workspace.cine_silver.peliculas
LIMIT 1
"""

try:
    spark.sql(mala)
    print("Se insertó. Esto NO debería pasar.")
except Exception as e:
    print("La tabla rechazó la fila, como debe:")
    print(str(e)[:400])

*Eso es un contrato de datos funcionando.* El error menciona la restricción
`presupuesto_positivo` por su nombre: por eso vale la pena ponerles nombres que se
entiendan solos.

---
## 7. Capa Gold: responder preguntas

Gold no es "Silver pero más bonito". Gold es **una tabla por pregunta de negocio**,
ya agregada, pensada para que un tablero la consulte sin hacer cálculos.

La primera pregunta: *¿cómo le fue a cada género, año por año?*

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.cine_gold.resumen_genero_anio
COMMENT 'Una fila por año y género: volumen, taquilla, retorno y nota media'
AS
SELECT
  year(fecha_estreno)                 AS anio,
  coalesce(genero, 'Sin clasificar')  AS genero,
  count(*)                            AS peliculas,
  sum(recaudacion_mundial)            AS recaudacion_mundial,
  sum(presupuesto)                    AS presupuesto,
  round(avg(try_divide(recaudacion_mundial, presupuesto)), 2) AS retorno_medio,
  round(avg(calificacion_imdb), 2)    AS nota_imdb_media
FROM workspace.cine_silver.peliculas
GROUP BY ALL

In [0]:
%sql
SELECT * FROM workspace.cine_gold.resumen_genero_anio
WHERE anio BETWEEN 2005 AND 2010
ORDER BY anio DESC, recaudacion_mundial DESC

Dos funciones que merecen comentario:

- **`coalesce(genero, 'Sin clasificar')`** — en Gold no se deja `NULL` en una
  dimensión. Un `NULL` en el filtro de un tablero desaparece de la vista, y nadie
  se entera de que faltan 269 películas.
- **`try_divide`** — divide y devuelve `NULL` si el divisor es cero, en vez de
  fallar. Aquí el `CHECK` ya garantiza presupuesto positivo, pero el hábito es
  bueno.

*Segunda pregunta: ¿qué estudios mandan?*

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.cine_gold.ranking_estudios
COMMENT 'Estudios con 20 o más películas, ordenados por taquilla mundial acumulada'
AS
SELECT
  estudio,
  count(*)                          AS peliculas,
  sum(recaudacion_mundial)          AS recaudacion_mundial,
  round(avg(calificacion_imdb), 2)  AS nota_imdb_media,
  rank() OVER (ORDER BY sum(recaudacion_mundial) DESC) AS puesto
FROM workspace.cine_silver.peliculas
WHERE estudio IS NOT NULL
GROUP BY estudio
HAVING count(*) >= 20

In [0]:
%sql
SELECT * FROM workspace.cine_gold.ranking_estudios ORDER BY puesto LIMIT 10

*Warner Bros. en lo alto, con Disney y Fox peleando el segundo puesto.* Una
pregunta de negocio respondida con una tabla de diez filas, construida sobre 3 183
películas que a su vez salieron de 3 201 filas crudas.

### 7.1 · Una vista también es gobierno

No todo el mundo necesita ver los presupuestos. Una vista que expone solo las
columnas públicas resuelve el caso sin duplicar datos: se dan permisos sobre la
vista, no sobre la tabla.

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.cine_gold.catalogo_publico
COMMENT 'Catálogo de películas sin información financiera. Apto para consumo general'
AS
SELECT titulo, fecha_estreno, genero, clasificacion, estudio, director,
       duracion_min, calificacion_imdb, calificacion_rt
FROM workspace.cine_silver.peliculas

In [0]:
%sql
SELECT * FROM workspace.cine_gold.catalogo_publico ORDER BY calificacion_imdb DESC LIMIT 5

---
## 8. Gobierno: quién, qué significa y qué pasó

Ya tenemos las tres capas. Ahora lo que convierte un montón de tablas en una
plataforma de datos: **saber qué hay, qué significa, quién puede verlo y qué le
pasó**.

### 8.1 · Documentar las columnas

Un comentario de columna es el texto que ve quien no estuvo en esta sesión. También
es lo que lee Genie para traducir preguntas en lenguaje natural a SQL.

In [0]:
COMENTARIOS = {
    "titulo":              "Título original de la película. Parte de la clave de negocio",
    "fecha_estreno":       "Fecha de estreno en salas. Parte de la clave de negocio",
    "presupuesto":         "Presupuesto de producción en dólares. Dato financiero confidencial",
    "recaudacion_mundial": "Taquilla mundial en dólares. NULL si el origen no la reportó",
    "calificacion_imdb":   "Nota media de IMDB, de 0 a 10",
}

for columna, texto in COMENTARIOS.items():
    spark.sql(f"ALTER TABLE workspace.cine_silver.peliculas "
              f"ALTER COLUMN {columna} COMMENT '{texto}'")
    print("Documentada:", columna)

In [0]:
%sql
DESCRIBE TABLE workspace.cine_silver.peliculas

### 8.2 · Etiquetas: clasificar sin tocar el dato

Las etiquetas (*tags*) son pares clave-valor que se pegan a esquemas, tablas y
columnas. Sirven para buscar y para aplicar políticas por categoría: *"dame todo lo
etiquetado como confidencial"*.

In [0]:
ETIQUETAS = [
    "ALTER TABLE workspace.cine_silver.peliculas "
    "SET TAGS ('capa' = 'silver', 'dominio' = 'cine', 'certificada' = 'si')",

    "ALTER TABLE workspace.cine_silver.peliculas ALTER COLUMN presupuesto "
    "SET TAGS ('sensibilidad' = 'confidencial')",

    "ALTER TABLE workspace.cine_gold.resumen_genero_anio "
    "SET TAGS ('capa' = 'gold', 'dominio' = 'cine')",
]

for sentencia in ETIQUETAS:
    spark.sql(sentencia)

print("Etiquetas aplicadas")

In [0]:
%sql
SELECT table_name, column_name, tag_name, tag_value
FROM workspace.information_schema.column_tags
WHERE schema_name = 'cine_silver'

### 8.3 · Permisos: cada capa a su público

El principio es el de mínimo privilegio: se concede lo que hace falta y nada más.
Un analista necesita leer Gold; no necesita entrar a Bronze.

Aquí concedemos lectura de `cine_gold` a **todos los usuarios de la cuenta**. En
una empresa esto iría a un grupo concreto, como `analistas_cine`.

In [0]:
PERMISOS = [
    "GRANT USE SCHEMA ON SCHEMA workspace.cine_gold TO `account users`",
    "GRANT SELECT     ON SCHEMA workspace.cine_gold TO `account users`",
]

for sentencia in PERMISOS:
    spark.sql(sentencia)
    print("Concedido:", sentencia.split(" ON ")[0])

In [0]:
%sql
SHOW GRANTS ON SCHEMA workspace.cine_gold

*Y lo contrario también es gobierno:* comprobar que la capa cruda **no** quedó
abierta a nadie.

In [0]:
%sql
SHOW GRANTS ON SCHEMA workspace.cine_bronze

### 8.5 · El inventario: qué tengo, en una consulta

`information_schema` es el catálogo del catálogo. Responde la pregunta que casi
ninguna empresa sabe contestar: **¿qué tablas tenemos y para qué son?**

In [0]:
%sql
SELECT table_schema AS capa, table_name AS tabla, table_type AS tipo, comment
FROM workspace.information_schema.tables
WHERE table_schema IN ('cine_bronze', 'cine_silver', 'cine_gold')
ORDER BY capa, tabla

*Esa consulta es el inventario completo de lo que acabamos de construir*, con la
descripción de cada pieza. Y se mantiene solo: nadie tiene que actualizar un Excel.

### 8.6 · Qué le pasó a la tabla

Lo de la Sesión 1, aplicado aquí: el historial registra cada operación sobre la
tabla Silver, con su hora.

In [0]:
hist = (spark.sql("DESCRIBE HISTORY workspace.cine_silver.peliculas")
             .select("version", "timestamp", "operation")
             .orderBy("version"))

display(hist)

---
## 9. Tu turno

Tres ejercicios sobre lo que acabas de construir. No hay celdas resueltas: la idea
es que escribas tú el SQL.

| # | Ejercicio | Pista |
|---|---|---|
| 1 | Crea una tabla Gold `peliculas_rentables` con las 50 películas de mejor retorno (taquilla ÷ presupuesto), contando solo las que tengan más de 10 000 votos en IMDB | `try_divide`, `WHERE`, `ORDER BY`, `LIMIT` |
| 2 | Añade una regla de calidad nueva: rechazar las películas que no tengan ni género ni director. ¿Cuántas filas más caen en cuarentena? | Modifica `peliculas_evaluadas` y vuelve a correr desde 5.1 |
| 3 | Documenta y etiqueta tu tabla nueva como en 8.1 y 8.2, y concede permiso de lectura sobre ella | `COMMENT`, `SET TAGS`, `GRANT` |

> **Pregunta para pensar, sin código:** las 16 películas con fecha futura están en
> cuarentena. ¿Quién debería decidir si se corrigen a mano, si se descartan o si se
> cambia la regla? Esa decisión no es técnica, y por eso el gobierno de datos no es
> solo una cuestión de permisos.

---
## 10. Limpieza

Borra todo lo que creó esta actividad: los tres esquemas con sus tablas, la vista,
la función de enmascarado y el volumen.

> Ejecútala al terminar, y también **antes** de volver a correr el notebook desde
> cero.

In [0]:
spark.sql("DROP VOLUME IF EXISTS workspace.cine_bronze.aterrizaje")

for esquema in ("cine_gold", "cine_silver", "cine_bronze"):
    spark.sql(f"DROP SCHEMA IF EXISTS workspace.{esquema} CASCADE")
    print("Eliminado: workspace." + esquema)

---
## 11. Lo que acabamos de hacer

| Capa | Qué guarda | Qué decisión toma |
|---|---|---|
| **Bronze** | El JSON tal como llegó, más de dónde y cuándo vino | Ninguna. Solo conserva |
| **Silver** | 3 183 películas tipadas, con contrato de datos | Qué fila es válida |
| **Cuarentena** | Las 18 que no pasaron, con el motivo | Qué fila hay que revisar |
| **Gold** | Agregados por género, año y estudio | Qué pregunta responde el negocio |

Y el gobierno que aplicamos encima:

| Herramienta | Para qué sirvió |
|---|---|
| Un esquema por capa | Dar permisos distintos a cada público |
| `COMMENT` en esquema, tabla y columna | Que se entienda sin preguntarle a nadie |
| `SET TAGS` | Clasificar lo confidencial y lo certificado |
| `NOT NULL`, `CHECK`, `PRIMARY KEY` | Que la tabla rechace lo que no cumple |
| Tabla de cuarentena | Que un dato malo no se pierda ni contamine |
| `GRANT` y `SHOW GRANTS` | Mínimo privilegio, y poder demostrarlo |
| Máscara de columna | Que el dato sensible se proteja solo |
| `information_schema` | Saber qué tenemos sin mantener un inventario a mano |
| `DESCRIBE HISTORY` y linaje | Saber qué pasó y de dónde viene cada número |

**La frase para llevarse:** las capas ordenan el dato, el gobierno lo hace
confiable. Sin lo segundo, lo primero es solo una carpeta con mejor nombre.